In [ ]:
import pandas as pd, os
BASE='/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
CLEAN_DIR=f'{BASE}/amia/clean_data'; FEATURE_DIR=f'{BASE}/amia/feature'; MATRIX_DIR=f'{BASE}/amia/matrix'
os.makedirs(MATRIX_DIR, exist_ok=True)
N_TOTAL=267747
EHR_TABLES=['lab','cond','drug','measurement','observation']

tf = 6   # 只重建 6m

a_pos=pd.read_csv(f'{CLEAN_DIR}/clean_positive_{tf}.csv'); a_pos['IsPositive']=1
a_neg=pd.read_csv(f'{CLEAN_DIR}/clean_negative_anchor_{tf}.csv'); a_neg['IsPositive']=0
anchors=pd.concat([a_pos,a_neg],ignore_index=True)[['person_id','IsPositive']]
assert len(anchors)==N_TOTAL

matrix=anchors.copy()
per_tbl={}
for tbl in EHR_TABLES:
    df=pd.read_parquet(f'{FEATURE_DIR}/{tbl}_features_{tf}.parquet')
    df=df.drop(columns=[c for c in df.columns if c.lower() in ('ispositive','label','is_positive')], errors='ignore')
    assert len(df)==N_TOTAL, f"{tbl} 长度异常: {len(df)}"
    per_tbl[tbl]=df.shape[1]-1
    matrix=matrix.merge(df, on='person_id', how='left')
    print(f"  [+] {tbl}: +{df.shape[1]-1} features → {matrix.shape}")

matrix.to_parquet(f'{MATRIX_DIR}/matrix_ehr_{tf}.parquet', index=False)
print(f"\n[saved] matrix_ehr_{tf}.parquet  shape={matrix.shape}  总特征={matrix.shape[1]-2}")
print("  各模态特征数:", per_tbl)

# 验证:condition 现在应 ~50 列、负例非全0
cond_cols=[c for c in matrix.columns if c.startswith('cond')]
neg=matrix[matrix['IsPositive']==0]
print(f"  condition 列={len(cond_cols)}  负例 condition 全0比例={(neg[cond_cols].fillna(0).abs().sum(axis=1)==0).mean():.3f}")

In [ ]:
import pandas as pd
import numpy as np

BUCKET = 'fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf'
FEATURE_DIR = f'gs://{BUCKET}/amia/feature'
CLEAN_DIR = f'gs://{BUCKET}/amia/clean_data'

N_TOTAL = 267747
EHR_TABLES = ['lab', 'cond', 'drug', 'measurement', 'observation']

def build_matrix(timeframe: int):
    print(f"\n{'='*60}\nTIMEFRAME = {timeframe}\n{'='*60}")
    
    # Anchor
    a_pos = pd.read_csv(f'{CLEAN_DIR}/clean_positive_{timeframe}.csv')
    a_neg = pd.read_csv(f'{CLEAN_DIR}/clean_negative_anchor_{timeframe}.csv')
    a_pos['IsPositive'] = 1
    a_neg['IsPositive'] = 0
    anchors = pd.concat([a_pos, a_neg], ignore_index=True)[['person_id', 'IsPositive']]
    assert len(anchors) == N_TOTAL
    print(f"[Anchor] {len(anchors):,}")
    
    matrix = anchors.copy()
    for tbl in EHR_TABLES:
        df = pd.read_parquet(f'{FEATURE_DIR}/{tbl}_features_{timeframe}.parquet')
        # 丢掉所有可能与 anchor 冲突的列（保留 person_id）
        drop_cols = [c for c in df.columns if c.lower() in ('ispositive', 'label', 'is_positive')]
        if drop_cols:
            df = df.drop(columns=drop_cols)
        assert len(df) == N_TOTAL, f"{tbl} 长度异常: {len(df)}"
        matrix = matrix.merge(df, on='person_id', how='left')
        print(f"  [+] {tbl}: +{df.shape[1]-1} features → {matrix.shape}")
    
    matrix_ehr = matrix.copy()
    
    surv = pd.read_parquet(f'{FEATURE_DIR}/survey_features_{timeframe}.parquet')
    drop_cols = [c for c in surv.columns if c.lower() in ('ispositive', 'label', 'is_positive')]
    if drop_cols:
        surv = surv.drop(columns=drop_cols)
    assert len(surv) == N_TOTAL, f"survey 长度异常: {len(surv)}"
    matrix_full = matrix.merge(surv, on='person_id', how='left')
    print(f"  [+] survey: +{surv.shape[1]-1} features → {matrix_full.shape}")
    
    print(f"\n[Final]")
    print(f"  ehr_{timeframe}:      {matrix_ehr.shape} | features = {matrix_ehr.shape[1]-2}")
    print(f"  ehr_surv_{timeframe}: {matrix_full.shape} | features = {matrix_full.shape[1]-2}")
    print(f"  Label: pos={(matrix_ehr['IsPositive']==1).sum():,} | neg={(matrix_ehr['IsPositive']==0).sum():,}")
    
    return matrix_ehr, matrix_full


matrices = {}
for tf in [6, 12, 24]:
    matrices[f'ehr_{tf}'], matrices[f'ehr_surv_{tf}'] = build_matrix(tf)

print("\n✅ 6 张表都在 matrices dict 里:")
for k, v in matrices.items():
    print(f"  matrices['{k}']: {v.shape}")

In [ ]:
BUCKET = 'fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf'
MATRIX_DIR = f'gs://{BUCKET}/amia/matrix'

for k, df in matrices.items():
    path = f'{MATRIX_DIR}/matrix_{k}.parquet'
    df.to_parquet(path, index=False)
    print(f"✓ matrix_{k}.parquet  {df.shape}")

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              precision_recall_curve)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_PRECISIONS = [0.30, 0.50, 0.70]

BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric=["aucpr", "auc"],
    eta=0.03,
    min_child_weight=16,
    subsample=0.75,
    colsample_bytree=0.70,
    reg_lambda=3.0,
    reg_alpha=0.4,
    tree_method="hist",
)

SPW_MULS = [0.5, 0.75, 1.0]
DEPTHS = [4, 5]


def pick_by_precision(prec, rec, thr, target):
    idx = np.where(prec[:-1] >= target)[0]
    if len(idx) == 0:
        return None
    j = idx[np.argmax(rec[idx])]
    return float(thr[j]), float(prec[j]), float(rec[j])


def train_once(X, y, spw_mul, depth, label=""):
    """单次训练，返回模型 + 验证集预测 + 指标"""
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
    base_spw = float(neg / max(pos, 1))
    
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dva = xgb.DMatrix(X_va, label=y_va)
    
    params = BASE_PARAMS.copy()
    params["scale_pos_weight"] = base_spw * spw_mul
    params["max_depth"] = depth
    
    print(f"  [{label}] spw_mul={spw_mul} depth={depth} → spw={base_spw*spw_mul:.2f}")
    bst = xgb.train(
        params, dtr, num_boost_round=5000,
        evals=[(dtr, "train"), (dva, "valid")],
        early_stopping_rounds=200,
        verbose_eval=200,
    )
    
    proba = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1))
    prauc = average_precision_score(y_va, proba)
    auc = roc_auc_score(y_va, proba)
    print(f"  [{label}] PR-AUC={prauc:.4f} | AUC={auc:.4f} | best_iter={bst.best_iteration}")
    return bst, proba, y_va, prauc, auc


# ============================================================
# Step 1: ehr_6m 完整 grid search
# ============================================================
print(f"\n{'='*70}\nSTEP 1: Grid search on ehr_6m\n{'='*70}")

df = matrices['ehr_6']
y = df['IsPositive']
X = df.drop(columns=['person_id', 'IsPositive'])
print(f"X shape: {X.shape} | pos={int(y.sum()):,} | neg={int((y==0).sum()):,}\n")

grid_results = []
for mul in SPW_MULS:
    for md in DEPTHS:
        print(f"\n--- spw_mul={mul} depth={md} ---")
        bst, proba, y_va, prauc, auc = train_once(X, y, mul, md, f"grid")
        grid_results.append({
            'spw_mul': mul, 'depth': md,
            'pr_auc': prauc, 'auc': auc,
            'best_iter': bst.best_iteration,
        })

# 总结 grid 结果
grid_df = pd.DataFrame(grid_results)
print(f"\n{'='*70}\nGrid Search Summary (ehr_6m)\n{'='*70}")
print(grid_df.to_string(index=False))

# 选最佳
best_idx = grid_df['pr_auc'].idxmax()
BEST_SPW = grid_df.loc[best_idx, 'spw_mul']
BEST_DEPTH = int(grid_df.loc[best_idx, 'depth'])
print(f"\n Best: spw_mul={BEST_SPW}, depth={BEST_DEPTH}, PR-AUC={grid_df.loc[best_idx, 'pr_auc']:.4f}")

In [ ]:
# ============================================================
# Step 2: 用 best params 跑剩下 5 个 matrix
# ============================================================
BEST_SPW = 0.5
BEST_DEPTH = 5

print(f"\n{'='*70}\nSTEP 2: Final runs with spw_mul={BEST_SPW}, depth={BEST_DEPTH}\n{'='*70}")

# ehr_6m 的结果从 Step 1 拿（spw_mul=0.5, depth=5 的那次已经训练过，但模型对象没保存）
# 为了一致性，所有 6 个 matrix 都重跑一遍，确保都是最终参数
all_results = []

for tf in [6, 12, 24]:
    for ver in ['ehr', 'ehr_surv']:
        label = f'{ver}_{tf}m'
        print(f"\n{'='*70}\n{label}\n{'='*70}")
        
        df = matrices[f'{ver}_{tf}']
        y = df['IsPositive']
        X = df.drop(columns=['person_id', 'IsPositive'])
        print(f"X shape: {X.shape} | pos={int(y.sum()):,} | neg={int((y==0).sum()):,}\n")
        
        bst, proba, y_va, prauc, auc = train_once(X, y, BEST_SPW, BEST_DEPTH, label)
        
        # Operating points
        prec, rec, thr = precision_recall_curve(y_va, proba)
        op_points = {}
        print(f"\n[Operating Points]")
        for tp in TARGET_PRECISIONS:
            picked = pick_by_precision(prec, rec, thr, tp)
            if picked:
                t, P, R = picked
                print(f"  Precision={tp:.2f} → Recall={R:.3f}, thr={t:.4f}")
                op_points[f'P={tp:.2f}_R'] = R
            else:
                print(f"  Precision={tp:.2f} → 无满足阈值")
                op_points[f'P={tp:.2f}_R'] = None
        
        beta = 0.5
        f = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-12)
        j = int(np.argmax(f[:-1]))
        print(f"  F0.5 best     → P={prec[j]:.3f}, R={rec[j]:.3f}, thr={thr[j]:.4f}")
        
        all_results.append({
            'label': label,
            'pr_auc': prauc,
            'auc': auc,
            'best_iter': int(bst.best_iteration),
            **op_points,
            'F05_P': float(prec[j]),
            'F05_R': float(rec[j]),
        })


# ============================================================
# 总结对比
# ============================================================
print(f"\n\n{'='*70}\nFINAL COMPARISON\n{'='*70}")
df_results = pd.DataFrame(all_results)
display_cols = ['label', 'pr_auc', 'auc', 'P=0.30_R', 'P=0.50_R', 'P=0.70_R', 'F05_P', 'F05_R']
print(df_results[display_cols].to_string(index=False))

print(f"\n[Survey 增益]")
for tf in [6, 12, 24]:
    ehr = next(r for r in all_results if r['label'] == f'ehr_{tf}m')
    surv = next(r for r in all_results if r['label'] == f'ehr_surv_{tf}m')
    print(f"  {tf}m: ΔPR-AUC = {surv['pr_auc']-ehr['pr_auc']:+.4f} | "
          f"ΔAUC = {surv['auc']-ehr['auc']:+.4f}")

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              precision_recall_curve)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_PRECISIONS = [0.30, 0.50, 0.70]
N_BOOTSTRAP = 1000
BEST_SPW = 0.5
BEST_DEPTH = 5

BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric=["aucpr", "auc"],
    eta=0.03,
    min_child_weight=16,
    subsample=0.75,
    colsample_bytree=0.70,
    reg_lambda=3.0,
    reg_alpha=0.4,
    tree_method="hist",
    max_depth=BEST_DEPTH,
)


def pick_by_precision(prec, rec, thr, target):
    idx = np.where(prec[:-1] >= target)[0]
    if len(idx) == 0:
        return None
    j = idx[np.argmax(rec[idx])]
    return float(thr[j]), float(prec[j]), float(rec[j])


def bootstrap_metrics(y_true, proba, n_iter=1000, seed=42):
    """对 (y_true, proba) 做 bootstrap，返回各指标的 95% CI"""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    
    metrics = {'pr_auc': [], 'auc': [],
               'R@P=0.30': [], 'R@P=0.50': [], 'R@P=0.70': []}
    
    for i in range(n_iter):
        idx = rng.randint(0, n, size=n)
        y_b = y_true[idx]
        p_b = proba[idx]
        
        # 跳过没有 pos 或 neg 的 bootstrap 样本（罕见）
        if len(np.unique(y_b)) < 2:
            continue
        
        metrics['pr_auc'].append(average_precision_score(y_b, p_b))
        metrics['auc'].append(roc_auc_score(y_b, p_b))
        
        prec, rec, thr = precision_recall_curve(y_b, p_b)
        for tp in TARGET_PRECISIONS:
            picked = pick_by_precision(prec, rec, thr, tp)
            metrics[f'R@P={tp:.2f}'].append(picked[2] if picked else np.nan)
    
    ci = {}
    for k, vals in metrics.items():
        arr = np.array([v for v in vals if not np.isnan(v)])
        if len(arr) > 0:
            ci[f'{k}_mean'] = arr.mean()
            ci[f'{k}_lo'] = np.percentile(arr, 2.5)
            ci[f'{k}_hi'] = np.percentile(arr, 97.5)
        else:
            ci[f'{k}_mean'] = ci[f'{k}_lo'] = ci[f'{k}_hi'] = np.nan
    return ci


# ============================================================
# 跑 6 个模型，保存 proba + y_va，再 bootstrap
# ============================================================
predictions = {}  # label -> (y_va, proba)
all_results = []

for tf in [6, 12, 24]:
    for ver in ['ehr', 'ehr_surv']:
        label = f'{ver}_{tf}m'
        print(f"\n{'='*70}\n{label}\n{'='*70}")
        
        df = matrices[f'{ver}_{tf}']
        y = df['IsPositive']
        X = df.drop(columns=['person_id', 'IsPositive'])
        
        X_tr, X_va, y_tr, y_va = train_test_split(
            X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
        )
        neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
        base_spw = float(neg / max(pos, 1))
        
        params = BASE_PARAMS.copy()
        params["scale_pos_weight"] = base_spw * BEST_SPW
        
        dtr = xgb.DMatrix(X_tr, label=y_tr)
        dva = xgb.DMatrix(X_va, label=y_va)
        
        print(f"[Train] spw={base_spw*BEST_SPW:.2f} depth={BEST_DEPTH}")
        bst = xgb.train(
            params, dtr, num_boost_round=5000,
            evals=[(dtr, "train"), (dva, "valid")],
            early_stopping_rounds=200,
            verbose_eval=200,
        )
        proba = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1))
        
        # 保存
        predictions[label] = (y_va.values, proba)
        
        # 点估计
        prauc = average_precision_score(y_va, proba)
        auc = roc_auc_score(y_va, proba)
        print(f"\n[Point estimate] PR-AUC={prauc:.4f} | AUC={auc:.4f}")
        
        # Bootstrap CI
        print(f"[Bootstrap] {N_BOOTSTRAP} iterations ...")
        ci = bootstrap_metrics(y_va.values, proba, n_iter=N_BOOTSTRAP)
        print(f"  PR-AUC: {ci['pr_auc_mean']:.4f} [{ci['pr_auc_lo']:.4f}, {ci['pr_auc_hi']:.4f}]")
        print(f"  AUC:    {ci['auc_mean']:.4f} [{ci['auc_lo']:.4f}, {ci['auc_hi']:.4f}]")
        for tp in TARGET_PRECISIONS:
            k = f'R@P={tp:.2f}'
            print(f"  {k}: {ci[f'{k}_mean']:.4f} [{ci[f'{k}_lo']:.4f}, {ci[f'{k}_hi']:.4f}]")
        
        all_results.append({
            'label': label,
            'pr_auc': prauc,
            'auc': auc,
            **ci,
        })


# ============================================================
# 总结表
# ============================================================
print(f"\n\n{'='*70}\nFINAL TABLE WITH 95% CI\n{'='*70}")
df_results = pd.DataFrame(all_results)

def fmt_ci(row, m):
    return f"{row[f'{m}_mean']:.4f} [{row[f'{m}_lo']:.4f}, {row[f'{m}_hi']:.4f}]"

print(f"\n{'Model':<18} {'PR-AUC (95% CI)':<28} {'AUC (95% CI)':<28}")
for _, r in df_results.iterrows():
    print(f"{r['label']:<18} {fmt_ci(r,'pr_auc'):<28} {fmt_ci(r,'auc'):<28}")

print(f"\n{'Model':<18} {'R@P=0.30':<28} {'R@P=0.50':<28} {'R@P=0.70':<28}")
for _, r in df_results.iterrows():
    print(f"{r['label']:<18} "
          f"{fmt_ci(r,'R@P=0.30'):<28} "
          f"{fmt_ci(r,'R@P=0.50'):<28} "
          f"{fmt_ci(r,'R@P=0.70'):<28}")

# ============================================================
# Survey 增益 + 配对 bootstrap test
# ============================================================
print(f"\n{'='*70}\nPAIRED BOOTSTRAP TEST (Survey vs EHR-only)\n{'='*70}")
print("ΔPR-AUC 的 95% CI（在同一 bootstrap 样本上算两个模型的差）")

for tf in [6, 12, 24]:
    y_ehr, p_ehr = predictions[f'ehr_{tf}m']
    y_surv, p_surv = predictions[f'ehr_surv_{tf}m']
    assert (y_ehr == y_surv).all(), "y_va 不一致！"
    
    rng = np.random.RandomState(42)
    diffs_prauc = []
    diffs_auc = []
    n = len(y_ehr)
    for _ in range(N_BOOTSTRAP):
        idx = rng.randint(0, n, size=n)
        y_b = y_ehr[idx]
        if len(np.unique(y_b)) < 2:
            continue
        d_pr = (average_precision_score(y_b, p_surv[idx])
                - average_precision_score(y_b, p_ehr[idx]))
        d_auc = roc_auc_score(y_b, p_surv[idx]) - roc_auc_score(y_b, p_ehr[idx])
        diffs_prauc.append(d_pr)
        diffs_auc.append(d_auc)
    
    diffs_prauc = np.array(diffs_prauc)
    diffs_auc = np.array(diffs_auc)
    
    print(f"\n{tf}m:")
    print(f"  ΔPR-AUC: mean={diffs_prauc.mean():+.4f}, "
          f"95% CI=[{np.percentile(diffs_prauc, 2.5):+.4f}, {np.percentile(diffs_prauc, 97.5):+.4f}], "
          f"p(>0)={(diffs_prauc > 0).mean():.3f}")
    print(f"  ΔAUC:    mean={diffs_auc.mean():+.4f}, "
          f"95% CI=[{np.percentile(diffs_auc, 2.5):+.4f}, {np.percentile(diffs_auc, 97.5):+.4f}], "
          f"p(>0)={(diffs_auc > 0).mean():.3f}")